In [0]:
# ============================================
# Cell 1: Imports
# ============================================
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, current_timestamp
from delta.tables import DeltaTable

# ============================================
# Cell 2: Config
# ============================================
bronze_table = "workspace.default.bronze_user_activity"
silver_table = "workspace.default.silver_user_activity"
checkpoint_path = "/Volumes/workspace/default/ecommerce_data/checkpoints/silver_user_activity"

# event_id is the natural unique key for a single event
dedup_keys = ["event_id"]

# ============================================
# Cell 3: Clean rebuild — drop old table + checkpoint
# ============================================
# One-time cleanup since the table was created earlier with a wrong
# column (event_timestamp instead of timestamp_ms). Safe to do since
# Silver is fully derivable from Bronze — no unique data lives only here.
spark.sql(f"DROP TABLE IF EXISTS {silver_table}")
dbutils.fs.rm(checkpoint_path, recurse=True)

# ============================================
# Cell 4: Create Silver table with correct schema
# ============================================
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_table} (
    event_id STRING,
    user_id STRING,
    session_id STRING,
    event_type STRING,
    product_id STRING,
    search_query STRING,
    amount DOUBLE,
    device_type STRING,
    geo_country STRING,
    timestamp_ms LONG,
    kafka_timestamp TIMESTAMP,
    ingested_at TIMESTAMP,
    processed_timestamp TIMESTAMP
)
USING DELTA
""")

# ============================================
# Cell 5: Read Bronze as a stream
# ============================================
bronze_stream = (
    spark.readStream
    .format("delta")
    .table(bronze_table)
)

# ============================================
# Cell 6: Merge logic per micro-batch
# ============================================
def upsert_to_silver(microBatchDF: DataFrame, batch_id: int):
    # Drop duplicates within the batch first — Kafka's at-least-once
    # delivery can redeliver the same event_id more than once per batch
    deduped_batch = microBatchDF.dropDuplicates(dedup_keys)

    # Keep only columns Silver needs; drop Kafka plumbing (partition, offset)
    deduped_batch = deduped_batch.select(
        "event_id", "user_id", "session_id", "event_type", "product_id",
        "search_query", "amount", "device_type", "geo_country",
        "timestamp_ms", "kafka_timestamp", "ingested_at"
    ).withColumn("processed_timestamp", current_timestamp())

    silver_delta_table = DeltaTable.forName(spark, silver_table)

    (
        silver_delta_table.alias("target")
        .merge(
            deduped_batch.alias("source"),
            "target.event_id = source.event_id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Batch {batch_id}: {deduped_batch.count()} deduplicated rows processed")

# ============================================
# Cell 7: Start the stream
# ============================================
query = (
    bronze_stream.writeStream
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)   # Free Edition constraint — see our earlier discussion on scheduled micro-batch
    .start()
)

query.awaitTermination()

# ============================================
# Cell 8: Verify
# ============================================
print(f"Silver rows: {spark.table(silver_table).count()}")
spark.table(silver_table).show(10, truncate=False)

In [0]:
%sql
-- Confirm Bronze actually has data to pull from
SELECT COUNT(*) FROM workspace.default.bronze_user_activity;

-- Confirm event_id is truly unique — if not, dedup_keys needs a composite key instead
SELECT COUNT(*) AS total, COUNT(DISTINCT event_id) AS distinct_ids
FROM workspace.default.bronze_user_activity;